# MeshVTON — Çıkarım (TEMMUZ checkpoint'i, TEXTURE'LI)

Bu notebook **yalnız** 2026-07-06 tarihli `ckpt_004000.pt` içindir: 4000 adım, texture'lı
appearance ref + %70 VITON-HD / %30 sentetik karışımı. Dissertation raporundaki Fig 2/3/4
bu sistemden geldi.

**Ana `MeshVTON_inference.ipynb` ile karıştırma.** Oradaki `final.pt` dokusuz sentetik-only
1000 adımla eğitildi ve gri çıktı üretir. İkisi farklı sistemlerdir.

`use_texture=True` bu notebook'a ÖZELDİR — checkpoint texture'lı ref görmeye alışkın,
gri ref verilirse dağıtım-dışı girdi alır. Projenin kalıcı NO-TEXTURE kuralı diğer her
yerde geçerlidir (`build_conditioning` varsayılanı `False`).

In [ ]:
#@title 1) Kurulum (train notebook'uyla aynı — pyrender/HMR2/IDM-VTON preprocess)
import os
if not os.path.exists('/content/MeshVTON'):
    !git clone https://github.com/SerhanTelatar/MeshVTON /content/MeshVTON
%cd /content/MeshVTON
!git pull

!pip -q install "diffusers>=0.34" "peft>=0.14" lpips einops sentencepiece trimesh smplx pyrender onnxruntime
!pip -q uninstall -y pyopengl PyOpenGL-accelerate > /dev/null 2>&1
!pip -q install "git+https://github.com/mmatl/pyopengl.git"
import importlib.util
if importlib.util.find_spec('hmr2') is None:
    !pip -q install "git+https://github.com/shubham-goel/4D-Humans.git"
!apt-get -qq install -y libglu1-mesa libosmesa6 > /dev/null 2>&1
import subprocess
_probe = subprocess.run(["python", "-c",
    'import os;os.environ["PYOPENGL_PLATFORM"]="egl";'
    'import pyrender;r=pyrender.OffscreenRenderer(16,16);r.delete();print("egl-ok")'],
    capture_output=True, text=True)
os.environ['PYOPENGL_PLATFORM'] = 'egl' if 'egl-ok' in _probe.stdout else 'osmesa'
print('GL platform:', os.environ['PYOPENGL_PLATFORM'])

if not os.path.exists('/content/IDM-VTON'):
    !git clone -q https://github.com/yisol/IDM-VTON /content/IDM-VTON
import shutil
from huggingface_hub import hf_hub_download
for repo_path in ('humanparsing/parsing_atr.onnx',
                  'humanparsing/parsing_lip.onnx',
                  'openpose/ckpts/body_pose_model.pth'):
    local = f'/content/IDM-VTON/ckpt/{repo_path}'
    if not (os.path.exists(local) and os.path.getsize(local) > 1_000_000):
        os.makedirs(os.path.dirname(local), exist_ok=True)
        shutil.copy(hf_hub_download('yisol/IDM-VTON', repo_path), local)
    assert os.path.getsize(local) > 1_000_000, f'bozuk indirme: {local}'

from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('Kurulum OK')

In [ ]:
#@title 2) Veri + TEMMUZ checkpoint'i
import os, sys, pathlib
sys.path.insert(0, '/content/MeshVTON/v2')
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/MeshVTON'

!mkdir -p /content/MeshVTON/data
!unzip -q -n $D/garments_3d.zip -d /content/MeshVTON/data
!mkdir -p /content/MeshVTON/data/raw
!unzip -q -n $D/images.zip -d /content/MeshVTON/data/raw

!mkdir -p /content/MeshVTON/checkpoints/pretrained/smplx
!cp $D/smplx/SMPLX_NEUTRAL.* /content/MeshVTON/checkpoints/pretrained/smplx/
os.environ['SMPLX_MODEL_DIR'] = '/content/MeshVTON/checkpoints/pretrained/smplx'

import glob, shutil
from meshvton2.conditioning.body import _patch_torch_load_weights_only
import torch as _torch; _patch_torch_load_weights_only(_torch)
from hmr2.models import download_models
from hmr2.configs import CACHE_DIR_4DHUMANS
download_models(CACHE_DIR_4DHUMANS)
smpl_dir = f"{CACHE_DIR_4DHUMANS}/data/smpl"; os.makedirs(smpl_dir, exist_ok=True)
cands = glob.glob(f'{D}/smpl/*neutral*lbs*.pkl') + glob.glob(f'{D}/smpl/SMPL_NEUTRAL.pkl')
assert cands, "SMPL neutral pkl yok -> Drive/MeshVTON/smpl/"
shutil.copy(cands[0], f"{smpl_dir}/SMPL_NEUTRAL.pkl")

# TEMMUZ checkpoint'i — stage1/ ALTINDA DEĞİL (yeni eğitim rotasyonu orayı siliyor)
CHECKPOINT = f"{D}/v2_outputs/stage1_july/ckpt_004000.pt"  #@param {type:"string"}
assert os.path.exists(CHECKPOINT), (
    f"bulunamadi: {CHECKPOINT}\n"
    "Drive copunden geri yukleyip stage1_july/ altina tasidiniz mi?")
print('checkpoint:', CHECKPOINT, f"({os.path.getsize(CHECKPOINT)/1e6:.0f} MB)")

from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

In [ ]:
#@title 3) Kişi + giysi seç, HİZALAMAYI kontrol et (difüzyon yok, ~1 dk)
# Temmuz checkpoint'i ESKİ koşullama rejiminde eğitildi (hang +0.06, ölçek 1.0,
# person_square_bbox düzeltmesinden ÖNCE). Ağustos kalibrasyonu (-0.12 / 1.25) ise
# BUGÜNKÜ foto yoluna göre ölçüldü. Hangisinin bu checkpoint'e uyduğu belirsiz —
# tahmin etmiyoruz, silüeti gövdeye bindirip GÖZLE seçiyoruz.
HANG_PAD      = -0.12  #@param {type:"number"}
GARMENT_SCALE = 1.25   #@param {type:"number"}
PERSON_ID     = "00000_00"  #@param {type:"string"}
GARMENT_ID    = "upper_body__00047_Top"  #@param {type:"string"}

import numpy as np, yaml, pathlib
from PIL import Image
from IPython.display import display, Markdown
from meshvton2.conditioning.body import build_hmr2_backend
from meshvton2.conditioning.builder import PhotoView, assert_real_impl, build_conditioning
from meshvton2.conditioning.garment import load_garment_asset
from meshvton2.conditioning.person import PersonPreprocessor, person_square_bbox
from meshvton2.eval.golden_set import load_manifest

REPO = pathlib.Path('/content/MeshVTON')
assert_real_impl()
base = yaml.safe_load((REPO/'v2/configs/base.yaml').read_text())
cfg  = yaml.safe_load((REPO/'v2/configs/eval.yaml').read_text())
size = (base['resolution']['height'], base['resolution']['width'])
man  = load_manifest(REPO/cfg['golden_manifest'])
GARMENTS_ROOT = REPO/base['paths']['garments_root']

g = globals()
if 'MV_PREP' not in g:
    g['MV_PREP'] = PersonPreprocessor('/content/IDM-VTON')
    g['MV_HMR2'] = build_hmr2_backend()
prep, hmr2 = g['MV_PREP'], g['MV_HMR2']

person = {p.id: p for p in man.persons}[PERSON_ID]
garment = {x.id: x for x in man.garments}[GARMENT_ID]
# TEXTURE'LI yükleme: bu checkpoint'in beklediği girdi bu
asset = load_garment_asset(GARMENTS_ROOT/garment.mesh,
                           texture_path=GARMENTS_ROOT/garment.texture if garment.texture else None,
                           garment_id=garment.id, allow_untextured=True)
print('giysi texture var mi:', asset.texture is not None,
      '' if asset.texture is not None else '  <-- YOK: use_texture faydasiz kalir')

pp = prep.process(man.root/person.image, size=size)
params = hmr2(pp.image, bbox=person_square_bbox(pp))
TH = 210; hh = round(TH*size[0]/size[1])
row = [('kişi', pp.image)]
for lbl, hp, sc in [('ESKİ (+0.06 / 1.00)', 0.06, 1.0), (f'KALİBRE ({HANG_PAD} / {GARMENT_SCALE})', HANG_PAD, GARMENT_SCALE)]:
    b = build_conditioning(pp.image, params, asset, PhotoView(), size=size, person_prep=pp,
                           hang_pad=hp, garment_scale=sc, use_texture=True)
    sil = b.control_depth_sil[2].numpy() > 0
    ov = pp.image.copy(); ov[sil] = (0.5*ov[sil] + 0.5*np.array([255,0,0])).astype(np.uint8)
    ref = ((b.appearance_ref.numpy().transpose(1,2,0)+1)/2*255).clip(0,255).astype(np.uint8)
    row += [(lbl, ov), ('ref', ref)]
strip = Image.new('RGB', (TH*len(row), hh), 'white')
for k,(_,a) in enumerate(row):
    strip.paste(Image.fromarray(a).convert('RGB').resize((TH,hh), Image.LANCZOS), (k*TH,0))
display(Markdown(' | '.join(l for l,_ in row)))
display(strip)
display(Markdown('**Kırmızı silüet hangi ayarda gövdeye daha iyi oturuyorsa onu seç; '
                 '`ref` panelleri RENKLİ olmalı** (gri ise texture yüklenmemiş).'))

In [ ]:
#@title 4) Üret — texture'lı ref ile (Temmuz rejimi)
GARMENT_IDS = "upper_body__00047_Top, upper_body__00111_Tshirt, upper_body__00126_Tshirt"  #@param {type:"string"}
STEPS = 28  #@param {type:"integer"}
SEED  = 0   #@param {type:"integer"}

from meshvton2.model.flux_tryon import FluxTryOnSampler
if 'MV_SAMPLER_JULY' not in g:
    print('FLUX + Temmuz checkpoint yükleniyor (~1 dk)...')
    g['MV_SAMPLER_JULY'] = FluxTryOnSampler(base['model']['flux_fill_repo'],
                                            checkpoint=CHECKPOINT, prompt=base['model']['prompt'])
sampler = g['MV_SAMPLER_JULY']

OUT = REPO/'v2/outputs/july_ckpt'; OUT.mkdir(parents=True, exist_ok=True)
by_gid = {x.id: x for x in man.garments}
gids = [x.strip() for x in GARMENT_IDS.split(',') if x.strip() in by_gid]
print(f'{len(gids)} giysi:', gids)

rows = []
for gid in gids:
    gm = by_gid[gid]
    a = load_garment_asset(GARMENTS_ROOT/gm.mesh,
                           texture_path=GARMENTS_ROOT/gm.texture if gm.texture else None,
                           garment_id=gm.id, allow_untextured=True)
    b = build_conditioning(pp.image, params, a, PhotoView(), size=size, person_prep=pp,
                           hang_pad=HANG_PAD, garment_scale=GARMENT_SCALE, use_texture=True)
    out = sampler.sample(b, steps=STEPS, seed=SEED, control_scale=1.0)
    Image.fromarray(out).save(OUT/f'{PERSON_ID}__{gid}.png')
    ref = ((b.appearance_ref.numpy().transpose(1,2,0)+1)/2*255).clip(0,255).astype(np.uint8)
    rows.append((gid, ref, out))
    print('bitti', gid)

grid = Image.new('RGB', (TH*3, hh*len(rows)), 'white')
for r,(gid,ref,out) in enumerate(rows):
    for c,a in enumerate([pp.image, ref, out]):
        grid.paste(Image.fromarray(a).convert('RGB').resize((TH,hh), Image.LANCZOS), (c*TH, r*hh))
display(Markdown('satır = giysi | sütunlar: **kişi** · **texture\'lı referans** · **çıktı**'))
display(grid)
display(Markdown('satır sırası: ' + ' · '.join(g_ for g_,_,_ in rows)))
print('kaydedildi →', OUT)